# 🕵️ Dia 48 — Falsas bandeiras, e uma declaração de atribuição que você consegue defender

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 48* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-48.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A sexta onda do IscaKit chegou com uma "assinatura bonita demais": um comentário que nomeia um grupo famoso de outro estado. Rafa desconfia. O Dr. Hélio pergunta: *"O que exatamente vocês podem afirmar para a delegacia?"*

Neste notebook você vai:

1. **Montar a linha do tempo de alegações** do Olympic Destroyer, em faixas, e exportar no formato do **TimelineJS**.
2. **Rodar o fluxograma "quem afirma o quê"** como função, para as três datas do original e para o caso Isca.
3. **Escrever as declarações** e passá-las num **verificador das cinco propriedades**.
4. **Gerar o índice do dossiê da Fase 3**.

🐍 Só biblioteca padrão. Este é o último dia da Fase 3.

## 1. As alegações do Olympic Destroyer

As sete linhas do original, mais quatro **para conferir nas fontes** (o original pede pelo menos onze, e que você confira cada data). Cada alegação vai numa das quatro faixas. Nenhum indivíduo é nomeado: o indiciamento de 2020 nomeia oficiais, e a regra do original é não pesquisá-los nem listá-los.

In [ ]:
# (data, quem alegou, alegação (nível), classe de evidência, falsificável?, faixa, fonte)
ALEGACOES_OD = [
    ("2018-02-09", "(evento)", "Perturbação durante a cerimônia de abertura", "n/a", "n/a",
     "Sem atribuição / evidência de falsificação", "https://blog.talosintelligence.com/2018/02/olympic-destroyer.html"),
    ("2018-02-12", "Cisco Talos", "Malware nomeado e descrito; sem atribuição", "Análise de malware", "n/a",
     "Sem atribuição / evidência de falsificação", "https://blog.talosintelligence.com/2018/02/olympic-destroyer.html"),
    ("2018-02", "Intezer", "Sobreposição de código com ferramentas do APT3 e APT10 (grupo)", "Código compartilhado",
     "Sim: código derivado do Mimikatz é público", "Apontou para a China", "(conferir: blog da Intezer, fev/2018)"),
    ("2018-02", "Recorded Future e outros", "Semelhanças com o Lazarus; recusaram-se a atribuir", "Código compartilhado",
     "Sim", "Apontou para a Coreia do Norte", "https://www.recordedfuture.com/research/olympic-destroyer-malware"),
    ("2018-02-24", "Autoridades dos EUA, via Washington Post", "GRU, com falsa bandeira para a Coreia do Norte (patrocinador)",
     "Não divulgada (fontes de governo)", "Não avaliável de fora", "Apontou para a Rússia",
     "https://www.washingtonpost.com/world/national-security/russian-spies-hacked-the-olympics-and-tried-to-make-it-look-like-north-korea-did-it-us-officials-say/2018/02/24/44b5468e-18f2-11e8-92c9-376b4fe57ff7_story.html"),
    ("2018-02-26", "Cisco Talos", "'Quem não foi responsável?': sobreposições não sustentam atribuição", "Análise de malware",
     "Expõe pistas falsificáveis", "Sem atribuição / evidência de falsificação",
     "https://blog.talosintelligence.com/2018/02/who-wasnt-responsible-for-olympic.html"),
    ("2018-03-08", "Kaspersky", "Rich header igual ao do Lazarus foi forjado", "Metadados do malware",
     "Esta descoberta expôs uma falsificação", "Sem atribuição / evidência de falsificação",
     "https://securelist.com/olympicdestroyer-is-here-to-trick-the-industry/84295/"),
    ("2018-10", "Talos (VB2018)", "Revisão longa: múltiplas pistas falsas (conferir)", "Análise de malware",
     "Expõe pistas falsificáveis", "Sem atribuição / evidência de falsificação",
     "https://www.virusbulletin.com/virusbulletin/2018/10/vb2018-paper-who-wasnt-responsible-olympic-destroyer/"),
    ("2020-10-19", "DOJ dos EUA", "Unidade 74455 do GRU (patrocinador e indivíduos)", "Processo judicial; inteligência",
     "Não avaliável de fora", "Apontou para a Rússia",
     "https://www.justice.gov/opa/pr/six-russian-gru-officers-charged-connection-worldwide-deployment-destructive-malware-and"),
    ("2020-10-19", "NCSC do Reino Unido", "Mesma atribuição à Unidade 74455 do GRU (conferir o comunicado)",
     "Inteligência", "Não avaliável de fora", "Apontou para a Rússia", "(conferir: ncsc.gov.uk, out/2020)"),
    ("(atual)", "MITRE ATT&CK", "S0365 Olympic Destroyer ligado ao G0034 Sandworm Team", "Agregação de relatórios",
     "Reflete as alegações acima", "Apontou para a Rússia", "https://attack.mitre.org/software/S0365/"),
]
FAIXAS = ["Apontou para a China", "Apontou para a Coreia do Norte", "Apontou para a Rússia",
          "Sem atribuição / evidência de falsificação"]
print(len(ALEGACOES_OD), "alegações;", len({a[5] for a in ALEGACOES_OD}), "faixas usadas")
pendentes = [a[0] + " " + a[1] for a in ALEGACOES_OD if "conferir" in (a[2] + a[6])]
print("A conferir na fonte antes de publicar:", pendentes)

### A linha do tempo em faixas

Uma coluna por mês, uma linha por faixa. Repare no padrão que o original descreve: **espalhado em fevereiro de 2018, concentrado depois dos comunicados de governo**.

In [ ]:
def linha_do_tempo(alegacoes, faixas):
    datas = sorted({a[0][:7] for a in alegacoes if a[0][0].isdigit()})
    print(f"{'faixa':<44}" + " ".join(d[2:] for d in datas))
    for faixa in faixas:
        celulas = []
        for d in datas:
            n = sum(1 for a in alegacoes if a[0].startswith(d) and a[5] == faixa)
            celulas.append(f"{('●' * n):^5}" if n else "  ·  ")
        print(f"{faixa[:43]:<44}" + " ".join(celulas))

linha_do_tempo(ALEGACOES_OD, FAIXAS)

In [ ]:
import csv

# Cabeçalho do modelo de planilha do TimelineJS (confira no modelo atual antes de colar).
COLUNAS_TL = ["Year", "Month", "Day", "Time", "End Year", "End Month", "End Day", "End Time",
              "Display Date", "Headline", "Text", "Media", "Media Credit", "Media Caption",
              "Media Thumbnail", "Type", "Group", "Background"]

def exportar_timeline(alegacoes, caminho):
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS_TL)
        w.writeheader()
        for data, quem, alegacao, classe, falsif, faixa, fonte in alegacoes:
            if not data[0].isdigit():
                continue                                  # "(atual)" não tem data: fica fora da linha do tempo
            partes = data.split("-")
            w.writerow({"Year": partes[0], "Month": partes[1] if len(partes) > 1 else "",
                        "Day": partes[2] if len(partes) > 2 else "",
                        "Headline": f"{quem}: {alegacao}",
                        "Text": f"Classe de evidência: {classe}. Falsificável? {falsif}.",
                        "Media": fonte if fonte.startswith("http") else "",
                        "Group": faixa})
    print("💾", caminho)

exportar_timeline(ALEGACOES_OD, "timeline-olympic-destroyer.csv")

## 2. O fluxograma "quem afirma o quê", como função

É o diagrama do passo 3 do original. As entradas são o que você **tem na data da declaração**; a saída é o ramo e o nível máximo de afirmação.

In [ ]:
def ramo(governo_oficial=False, so_imprensa_anonima=False, verificavel_por_mim=False,
         sinal_de_falsificacao=False, sobreposicoes_explicaveis=True):
    """Devolve a lista de passos percorridos e a orientação final."""
    passos = []
    if governo_oficial:
        passos.append("Governo atribuiu em comunicado oficial → relate como alegação DELES")
        if verificavel_por_mim:
            passos.append("Verificável por você → diga o que verificou e como, separadamente")
        else:
            passos.append("Não verificável → mantenha atribuída a eles; NÃO reescreva como fato")
        return passos
    if so_imprensa_anonima:
        passos.append("Só autoridades anônimas na imprensa → cite como alegação relatada e não verificada")
    if sinal_de_falsificacao:
        passos.append("Item técnico com sinal de falsificação → engano provável; quem enganou segue desconhecido; "
                      "NÃO nomeie grupo nem Estado")
    elif sobreposicoes_explicaveis:
        passos.append("Toda sobreposição explicável por código público/copiado → nenhum grupo ou Estado; "
                      "no máximo nível CLUSTER")
    else:
        passos.append("Sobreposição não trivial → afirmação de nível GRUPO pode ser defensável; gradue (Dia 47)")
    return passos

CENARIOS = {
    "2018-02-20 (Olympic Destroyer)": dict(),
    "2018-03-10 (Olympic Destroyer)": dict(so_imprensa_anonima=True, sinal_de_falsificacao=True),
    "2020-10-20 (Olympic Destroyer)": dict(governo_oficial=True, verificavel_por_mim=False),
    "2026-10-14 (CA-ISCA-01, sexta onda)": dict(sinal_de_falsificacao=True),
}
for data, entradas in CENARIOS.items():
    print(f"\n📅 {data}")
    for p in ramo(**entradas):
        print("   →", p)
    print("   → Escreva as cinco propriedades.")

## 3. O verificador das cinco propriedades

Toda declaração precisa de: **(1)** nível · **(2)** probabilidade e confiança · **(3)** classes de evidência · **(4)** alternativa mais forte · **(5)** o que mudaria. Mais as regras do checkpoint: nenhum indivíduo nomeado; alegação de governo atribuída a quem a fez; nada de Estado nomeado em fevereiro de 2018.

A detecção é por palavras-chave, então é **ajuda, não juiz**: leia cada alerta.

In [ ]:
import re

PROPRIEDADES = {
    "1 nível": r"\b(cluster|agrupamento|grupo|patrocinador|estado|indiv[ií]duo|ator capaz|nenhum grupo)\b",
    "2a probabilidade": r"\b(prov[áa]vel|improv[áa]vel|quase cert|possibilidade realista|chance remota|altamente)\w*",
    "2b confiança": r"\bconfian[çc]a\b",
    "3 classes de evidência": r"\b(c[óo]digo|infraestrutura|malware|metadad|certificado|servidor|tradecraft|"
                              r"hor[áa]rio|kit|rich header|comunicado|fontes?)\b",
    "4 alternativa mais forte": r"alternativa",
    "5 o que mudaria": r"\bmudari(a|am)\b|\bmudar esta avalia",
}
ESTADOS = r"\b(r[úu]ssia|russ[oa]|china|chin[êe]s|coreia|norte-coreano|gru)\b"

def verificar_declaracao(texto, data=""):
    baixo = texto.lower()
    alertas = [f"falta a propriedade {nome}" for nome, padrao in PROPRIEDADES.items()
               if not re.search(padrao, baixo)]
    # Nomes próprios de pessoa (heurística "Nome Sobrenome", ignorando organizações conhecidas)
    ignorar = {"Washington Post", "Cisco Talos", "Recorded Future", "Olympic Destroyer", "Unidade 74455",
               "Serra Azul", "Reino Unido", "Coreia do", "Dr. Hélio"}
    candidatos = [n for n in re.findall(r"\b[A-ZÁÉÍÓÚ][a-záéíóúãõç]+ [A-ZÁÉÍÓÚ][a-záéíóúãõç]+\b", texto)
                  if not any(n.startswith(i.split()[0]) and i in texto for i in ignorar)]
    if candidatos:
        alertas.append(f"possível nome de pessoa: {candidatos}")
    if data.startswith("2018-02") and re.search(ESTADOS, baixo):
        alertas.append("fev/2018 não pode nomear Estado como responsável")
    if re.search(r"\bgru\b", baixo) and data.startswith("2020"):
        if not re.search(r"governos?|atribuem|segundo|de acordo com", baixo):
            alertas.append("a alegação do GRU precisa ser atribuída aos governos que a fizeram")
        if re.search(r"(^|\. )a unidade 74455 do gru (realizou|conduziu|executou)", baixo):
            alertas.append("afirmação de governo reescrita como fato verificado por você")
    return alertas

def mostrar(nome, texto, data=""):
    a = verificar_declaracao(texto, data)
    print(f"{'✅' if not a else '⚠️ '} {nome}")
    for x in a:
        print("    -", x)

EXEMPLO_FEV = ("Avaliamos que a operação Olympic Destroyer foi conduzida por um ator capaz que plantou "
               "deliberadamente evidências enganosas (provável; confiança moderada). As sobreposições de código "
               "com ferramentas atribuídas a grupos ligados à China e ao Lazarus são, cada uma, explicáveis por "
               "código público ou copiado, e a evidência à nossa disposição não sustenta atribuição a nenhum grupo "
               "ou Estado conhecido. A alternativa mais forte é que uma dessas sobreposições seja genuína. A "
               "recuperação de ferramental ou infraestrutura não pública e exclusiva de um grupo conhecido mudaria "
               "esta avaliação.")
mostrar("Exemplo resolvido do original (20/fev/2018)", EXEMPLO_FEV, "2018-02-20")

> 💡 O exemplo do original cita "grupos ligados à **China**" e o verificador acusa. Leia o alerta: é falso positivo, porque a frase **relata** a direção da pista e nega a atribuição. O verificador não entende negação; você entende. É exatamente o tipo de alerta que se lê e se justifica.

In [ ]:
#@title Suas declarações (Olympic Destroyer) { display-mode: "form" }
declaracao_mar_2018 = "Avaliamos como altamente provável que a operação Olympic Destroyer incluiu evidências plantadas para enganar a atribuição. Confiança: alta, porque a Kaspersky mostrou que o Rich header igual ao do Lazarus é incoerente com o restante do binário. Autoridades dos EUA citadas anonimamente pelo Washington Post atribuem a operação a um Estado, mas não conseguimos verificar essa alegação. A evidência técnica à nossa disposição não identifica quem plantou as pistas, e não sustenta atribuição a nenhum grupo conhecido. A alternativa mais forte é que alguma sobreposição de código seja genuína. Ferramental ou infraestrutura não pública e exclusiva de um grupo mudaria esta avaliação."  #@param {type:"string"}
declaracao_out_2020 = "Os governos dos EUA e do Reino Unido atribuem a operação Olympic Destroyer à Unidade 74455 do GRU (nível patrocinador e indivíduos, que não nomeamos), com base em fontes de inteligência e processo judicial que não são públicos. Não verificamos essa alegação de forma independente. Do nosso lado, avaliamos como altamente provável que o malware tenha sido construído com pistas falsas. Confiança: alta, pela análise do Rich header. A alternativa mais forte à atribuição governamental é que outro ator tenha conduzido a operação, o que nenhum dado público sustenta. A divulgação das provas do processo mudaria a nossa capacidade de verificar."  #@param {type:"string"}

mostrar("Declaração de 10/mar/2018", declaracao_mar_2018, "2018-03-10")
mostrar("Declaração de 20/out/2020", declaracao_out_2020, "2020-10-20")

## 4. ➕ O caso Isca: a sexta onda

Alegações de `caso-continuo/dados/dia-48/alegacoes-iscakit.csv` (fictícias), na mesma estrutura e com quatro faixas.

In [ ]:
import io
import urllib.request

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-48/alegacoes-iscakit.csv")
EMBUTIDO = """data,quem_alegou,alegacao,nivel,classe_evidencia,falsificavel,faixa,fonte
2026-10-05,(evento),Primeira onda de SMS 'Pix bloqueado' contra a Serra Azul,n/a,n/a,n/a,Sem atribuição / evidência de falsificação,Reclamações da ouvidoria (caso VIGIA-2026-001)
2026-10-07,Vigia,Cinco domínios e dois servidores agrupados no cluster CA-ISCA-01,cluster,infraestrutura,"Médio: certificado com SAN compartilhado",Cluster CA-ISCA-01 (Vigia),Grafo de pivôs (Dia 40) e bundle STIX (Dia 43)
2026-10-08,Print anônimo em grupo,Anúncio de venda do 'IscaKit cooperativas',grupo,fonte externa,"Sim: print não verificado",Apontou para o vendedor do kit,Print encaminhado por terceiro
2026-10-10,Post de analista no LinkedIn,'Mesmo painel de um kit vendido no Telegram: caso resolvido',grupo,malware,"Sim: painel vem com o kit comprado",Apontou para o vendedor do kit,Post público (não reproduzido)
2026-10-12,Vigia,Informe CA-ISCA-01: sem afirmação de autoria,cluster,infraestrutura; tradecraft,n/a,Cluster CA-ISCA-01 (Vigia),Informe de uma página (Dia 46)
2026-10-14,(evento),Sexta onda com bloco de comentário 'assinando' em nome de um grupo de outro estado,n/a,n/a,n/a,Sem atribuição / evidência de falsificação,Captura da página em navegador isolado
2026-10-14,Comentário no código da sexta onda,Autoria atribuída a 'o grupo nomeado no comentário',grupo,malware (string),"Sim: texto trivial de plantar",Apontou para o grupo do outro estado,Captura da página em navegador isolado
2026-10-14,Vigia,"Bloco de 'assinatura' incoerente com cinco ondas anteriores: provável falsificação",cluster,malware (metadados e strings),Esta análise expôs provável falsificação,Sem atribuição / evidência de falsificação,Comparação das capturas das seis ondas
2026-10-15,Página de notícias regional,'Grupo do outro estado ataca cooperativas mineiras',grupo,não divulgada,Não avaliável de fora; repete o comentário plantado,Apontou para o grupo do outro estado,Reportagem (não reproduzida)
2026-10-16,Vigia,Declaração de atribuição CA-ISCA-01 (Dia 48),cluster,infraestrutura; tradecraft; malware,n/a,Cluster CA-ISCA-01 (Vigia),Declaração deste dia
2026-10-16,Cooperativa Serra Azul (via jurídico),Comunicação de fatos e artefatos à delegacia especializada; sem imputação de autoria,n/a,n/a,n/a,Sem atribuição / evidência de falsificação,Ofício da cooperativa (fictício)
"""
try:
    with urllib.request.urlopen(URL, timeout=10) as r:
        texto = r.read().decode("utf-8")
except Exception as erro:
    texto = EMBUTIDO
    print(f"Sem rede ({type(erro).__name__}): usando a cópia embutida.")

ALEGACOES_ISCA = [(l["data"], l["quem_alegou"], l["alegacao"], l["classe_evidencia"], l["falsificavel"],
                   l["faixa"], l["fonte"]) for l in csv.DictReader(io.StringIO(texto))]
FAIXAS_ISCA = ["Apontou para o vendedor do kit", "Apontou para o grupo do outro estado",
               "Cluster CA-ISCA-01 (Vigia)", "Sem atribuição / evidência de falsificação"]

# A linha do tempo do caso é por DIA (tudo acontece em outubro de 2026).
dias = sorted({a[0] for a in ALEGACOES_ISCA})
print(f"{'faixa':<42}" + " ".join(d[8:] for d in dias) + "   ← dias de out/2026")
for faixa in FAIXAS_ISCA:
    print(f"{faixa[:41]:<42}" + " ".join(("●" * sum(1 for a in ALEGACOES_ISCA if a[0] == d and a[5] == faixa)).center(2) or " ·" for d in dias))
exportar_timeline(ALEGACOES_ISCA, "timeline-iscakit.csv")

In [ ]:
#@title Declaração CA-ISCA-01 e parágrafo para o Dr. Hélio { display-mode: "form" }
declaracao_isca = "Avaliamos como provável que a sexta onda pertence ao cluster CA-ISCA-01. Confiança: moderada, porque o kit, o caminho de envio e o horário de disparo coincidem com as ondas anteriores, mas não há certificado nem servidor em comum. Avaliamos como provável que o bloco de comentários que cita o grupo nomeado no comentário foi plantado para desviar a atribuição. Confiança: moderada, porque ele é incoerente com cinco ondas anteriores sem assinatura e nomeia um grupo de alta visibilidade. A evidência à nossa disposição não sustenta atribuição a nenhum grupo ou pessoa. A alternativa mais forte é que o operador tenha passado a assinar por descuido ou vaidade. Dados cadastrais e registros de conexão obtidos por ordem judicial mudariam esta avaliação."  #@param {type:"string"}
paragrafo_helio = "A cooperativa pode comunicar à delegacia os fatos (datas e horários dos SMS, domínios, endereços IP, número de reclamações e de transferências não reconhecidas) e os artefatos coletados, com hash SHA-256 e registro de custódia, acompanhados desta avaliação identificada como avaliação técnica. Não deve indicar o grupo citado no comentário como suspeito, nem qualquer pessoa: o comentário é tratado como possível pista plantada. Os pedidos que chegam a pessoas cabem à autoridade policial e ao Judiciário."  #@param {type:"string"}

mostrar("Declaração CA-ISCA-01 (14/out/2026)", declaracao_isca, "2026-10-14")

problemas = []
if not re.search(r"hash|sha-?256", paragrafo_helio, re.I):
    problemas.append("o parágrafo não menciona hash/custódia dos artefatos")
if not re.search(r"avalia[çc][ãa]o", paragrafo_helio, re.I):
    problemas.append("o parágrafo não separa a avaliação dos fatos")
if re.search(r"\b(foi|é|são) (o|os) (autor|respons[áa]ve)", paragrafo_helio, re.I):
    problemas.append("o parágrafo imputa autoria")
print("✅ Parágrafo para o Dr. Hélio" if not problemas else "⚠️  Parágrafo: " + "; ".join(problemas))

## 5. 🎮 Quiz: alegação deles ou fato verificado?

Para cada frase, responda **1** se ela está corretamente escrita como **alegação atribuída**, **2** se ela apresenta como **fato** algo que você não verificou, ou **3** se ela **nomeia indevidamente** um ator.

In [ ]:
FRASES = [
    ("Os governos dos EUA e do Reino Unido atribuem a operação à Unidade 74455 do GRU.", "1",
     "Fato com fonte, sobre a alegação deles."),
    ("A Unidade 74455 do GRU realizou a operação Olympic Destroyer.", "2",
     "É a alegação dos governos reescrita como fato que você não verificou."),
    ("Em 20/fev/2018: o Olympic Destroyer foi obra da Coreia do Norte, como mostra o código do Lazarus.", "3",
     "Nomeia um Estado a partir de pista copiável; e era uma falsa bandeira."),
    ("Segundo a Kaspersky, o Rich header igual ao do Lazarus foi forjado.", "1", "Alegação atribuída à fonte."),
    ("A sexta onda do IscaKit foi feita pelo grupo nomeado no comentário do código.", "3",
     "Aceita a pista plantada como autoria. No Brasil, isso pode ir parar no art. 339 do CP."),
    ("O comentário da sexta onda cita um grupo de outro estado; avaliamos como provável que tenha sido plantado.", "1",
     "Relata o conteúdo do artefato e dá a avaliação calibrada."),
]
MODO_JOGO = True
acertos = 0
ROT = {"1": "alegação atribuída", "2": "fato não verificado", "3": "nomeia indevidamente"}
for n, (f, certo, porque) in enumerate(FRASES, start=1):
    print(f"\n[{n}/{len(FRASES)}] {f}")
    if MODO_JOGO:
        r = ""
        while r not in ROT:
            r = input("   1 atribuída · 2 fato não verificado · 3 nomeia indevidamente: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {ROT[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(FRASES)}")

## 6. 📁 O índice do dossiê da Fase 3

A tabela do passo 4 do original, somada aos artefatos da Operação Isca dos Dias 42 a 48. Preencha a anotação do passo 5 (teste com um não especialista), **sem nomear a pessoa**.

In [ ]:
#@title Anotação do passo 5 { display-mode: "form" }
teste_nao_especialista = "Li a declaração do CA-ISCA-01 para uma pessoa de fora da área; ela disse, sem perguntar, que estamos 'razoavelmente seguros de que é o mesmo grupo de golpe, mas não de quem é', e que a assinatura nova parece armação."  #@param {type:"string"}
precisou_revisar = False  #@param {type:"boolean"}

ORIGINAL = [
    (35, "Mapa de calor de densidade de subtécnicas; ficha de ID de técnica"),
    (36, "Camada do Volt Typhoon com evidências; mapa de calor de comparação"),
    (37, "Perfil do APT29; correspondência de aliases; ficha Diamond"),
    (38, "Mapa de calor de taxa de base; APT29 × APT28; linha do tempo de comportamento"),
    (39, "Tabela de enriquecimento de IOCs"),
    (40, "Grafo de pivôs de infraestrutura; registro de pivôs"),
    (41, "Registro de instalação; plano de coleta; mapa de calor de risco de fontes"),
    (42, "Mapas de calor de sites de vazamento com ressalvas"),
    (43, "Bundle STIX validado; captura do visualizer"),
    (44, "Registro de consultas TAXII; registro de decisão de compartilhamento"),
    (45, "Matriz ACH do NotPetya; checagem de premissas-chave"),
    (46, "Informe de uma página"),
    (47, "Grafo de evidências do Turla"),
    (48, "Linha do tempo do Olympic Destroyer; três declarações de atribuição"),
]
ISCA = [
    (42, "Resposta de cinco linhas ao conselho (postagens BR/financeiro, com ressalvas)"),
    (43, "cluster-iscakit.json (intrusion-set CA-ISCA-01, sem threat-actor)"),
    (44, "Três registros de decisão de compartilhamento; bundle marcado e limpo"),
    (45, "Matriz ACH do IscaKit; premissas-chave"),
    (46, "Informe CA-ISCA-01 para a Marta (PHIA, TLP:AMBER)"),
    (47, "Graduação de evidências do IscaKit; contrafactual no nível cluster"),
    (48, "Linha do tempo de alegações do caso; declaração CA-ISCA-01; parágrafo para o Dr. Hélio"),
]
indice = ["# Dossiê do caso — Fase 3 (Inteligência de ameaças)", "",
          "| Dia | Artefato (original) |", "|---|---|"]
indice += [f"| {d} | {a} |" for d, a in ORIGINAL]
indice += ["", "## ➕ Operação Isca (Dias 42 a 48)", "", "| Dia | Artefato |", "|---|---|"]
indice += [f"| {d} | {a} |" for d, a in ISCA]
indice += ["", "## Teste com um não especialista (passo 5)", "",
           teste_nao_especialista + (" Foi preciso revisar a declaração." if precisou_revisar else ""),
           "", "_Sem nomes de pessoas neste dossiê._"]
from pathlib import Path
Path("dossie-fase-3").mkdir(exist_ok=True)
Path("dossie-fase-3/INDICE.md").write_text("\n".join(indice) + "\n", encoding="utf-8")
Path("dossie-fase-3/declaracoes.md").write_text(
    "# Declarações de atribuição\n\n"
    f"## 20/fev/2018 (exemplo do original)\n{EXEMPLO_FEV}\n\n## 10/mar/2018\n{declaracao_mar_2018}\n\n"
    f"## 20/out/2020\n{declaracao_out_2020}\n\n## ➕ 14/out/2026 — CA-ISCA-01\n{declaracao_isca}\n\n"
    f"### Para o Dr. Hélio\n{paragrafo_helio}\n", encoding="utf-8")
print("\n".join(indice[:6]) + "\n...")
print("💾 dossie-fase-3/INDICE.md e dossie-fase-3/declaracoes.md")
try:
    from google.colab import files
    for n in ("dossie-fase-3/INDICE.md", "dossie-fase-3/declaracoes.md",
              "timeline-olympic-destroyer.csv", "timeline-iscakit.csv"):
        files.download(n)
except ImportError:
    pass

## ✅ Antes de fechar o Dia 48 (e a Fase 3)

- [ ] Linha do tempo com pelo menos 11 alegações conferidas na fonte, publicada no TimelineJS
- [ ] Declaração de fev/2018 sem Estado; de out/2020 atribuída aos governos e não apresentada como verificada
- [ ] Declaração CA-ISCA-01 no nível cluster, com as cinco propriedades
- [ ] Parágrafo para o Dr. Hélio sem imputação de autoria
- [ ] `dossie-fase-3/INDICE.md` com a anotação do teste com um não especialista
- [ ] Nenhum indivíduo nomeado em nenhum artefato

🥷 **Fase 3 concluída.** Rank `B` consolidado.

**Próximo (Dia 49, Fase 4):** uma caixa da agência Centro abriu o link no notebook corporativo `SA-CX-07`. A investigação sai da internet e entra no disco: manuseio de evidência, hash e cadeia de custódia.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>